Relevance Formula: asks, ``for the RELU value of the neuron in the l-th layer at the h-th height, how much do we impact the entirety of the RELU value of the next layer, summed across all heights"

$R_{\ell = \text{layer}, h = \text{height}} = \frac{\text{RELU}(V_{\ell, h})}{\sum_{h} \text{RELU}(V_{\ell+1, h})}$

In [74]:
import pandas as pd
import numpy as np
import random as rd

# Initialize the modern NumPy random generator
rng = np.random.default_rng(seed=42)


def weightMaker(n,m, l):
    #makes weight matrices of size n \times m 
    mtx = rng.exponential(scale=l, size=(n, m))
    return mtx

def biasMaker(n, mean, std):
    bias = rng.normal(loc=mean, scale=std, size=(n, 1))
    return bias

def RELU(weight, bias, data):
    A = (weight @ data) + bias
    for row in range(len(A)):
        if A[row] < 0:
            A[row] = 0
    return A

def rel(A, nextMatrix, nextBias): #in each row, how much does this layer impact the entirety of the next layer?
    relevance = []
    sumup = 0
    B =RELU(nextMatrix, nextBias, A)
    for row in range(len(A)):
        for i in range(len(nextMatrix)):
            sumup += B[i]
        if sumup != 0:
            relevance += [A[row] / sumup]
        else:
            relevance += [10000000000000000000000000]
    return relevance

#each sublist is a layer, each layer has [size of starting data (scalar), size requirement of next layer]
data = np.array([[1], [1], [-1]])
architecture = [[len(data),4], [4,2], [2,2], [2, 1]]
bookeeping = []
nextMatrix = 0

for i in range(len(architecture)):
    print('-----------------------')
    arch = architecture[i]
    if i == 0:
        w = weightMaker(arch[1], arch[0], 1)
        bias = biasMaker(arch[1], 100, 50)
    else:
        w = nextMatrix
        bias = nextBias
    
    data = RELU(w, bias, data)
    if i != len(architecture)-1:
        nextMatrix = weightMaker(architecture[i+1][1], architecture[i+1][0], 1)
        nextBias = biasMaker(architecture[i+1][1], 100, 50)
        relevance = rel(data, nextMatrix, nextBias)
    else:
        relevance = 0
    bookeeping += [w, bias, relevance, data]
    if i != len(architecture):
        print(f'V_{i+1} relevance is {relevance}')



-----------------------
V_1 relevance is [array([0.12732222]), array([0.09355748]), array([0.05134737]), array([0.01719105])]
-----------------------
V_2 relevance is [array([0.31228209]), array([0.3133812])]
-----------------------
V_3 relevance is [array([0.72444997]), array([0.49166383])]
-----------------------
V_4 relevance is 0
